# 08 · Adaptation and compositional generalization
### Can a solver reuse what it learned when the next task is different?

This final lesson of **vector_01** connects the entire sequence:

- **00–01:** Learn computations and mathematical expressions.
- **02–03:** Represent programs and compare hypotheses.
- **04–05:** Allocate search and learn priors over decisions.
- **06:** Represent structure and observable execution.
- **07:** Learn reusable language abstractions.
- **08:** Test whether any of that **transfers beyond familiar examples**.

**Research question:** When primitive operations are familiar but their **combinations** are not, can a search prior generalize? If the task distribution shifts, how much can we safely adapt it with a few solved examples?

We will implement a complete, reproducible *toy* experiment using a tiny program language and exact interpreter, and evaluate four kinds of held-out tasks, fixed candidate-execution budgets, false solutions that overfit demonstrations, online-style support-set updates, and failure when the grammar cannot express a solution.

**What this is not:** a claim about performance on ARC-AGI, a neural meta-learner, or a universal task-distribution study. The policy here is a transparent **smoothed bigram model of operator sequences**. That simplicity makes its failure modes interpretable and gives you a benchmark architecture to replace with lesson 05's task-conditioned neural policy and lesson 06's embeddings.

**Environment:** NumPy and Matplotlib only. Run top to bottom. Predict outcomes at each marked prompt before looking at the plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from collections import Counter

SEED=808
WIDTH=7
TOKENS=('R','L','F','N')
TO_IDX={a:i for i,a in enumerate(TOKENS)}
plt.rcParams.update({'figure.figsize':(9,3.8),'axes.grid':True,'grid.alpha':.19})
np.set_printoptions(precision=3,suppress=True)
print('Lesson 08: deterministic synthetic tasks, exact interpreter, no ML dependencies.')

## 1 · A concrete grammar and an exact execution oracle

A *program* is a sequence of unary operations acting on a seven-cell Boolean row:

- **R:** shift right, fill new left cell with zero.
- **L:** shift left, fill new right cell with zero.
- **F:** reverse the cell order.
- **N:** Boolean NOT.

We compose operations **left-to-right**: (R,N) means apply R, **then** N. Order often changes the resulting function. We allow sequences of length zero through five: \(1+4+4^2+\cdots+4^5=1365\) syntactically distinct programs.

In this tiny world, there are only \(2^7=128\) possible inputs. For **ground-truth equivalence and split construction only**, we can enumerate the complete function of each program. The actual search evaluation later will receive **only a few task demonstrations**.

**Predict:** Can two different sequences implement exactly the same function on all 128 inputs?

In [ ]:
ALL_INPUTS=((np.arange(1<<WIDTH,dtype=np.uint16)[:,None]>>np.arange(WIDTH))&1).astype(bool)

def step(x,op):
    x=np.asarray(x,dtype=bool)
    if op=='R':
        return np.concatenate([np.zeros_like(x[...,:1]),x[...,:-1]],axis=-1)
    if op=='L':
        return np.concatenate([x[...,1:],np.zeros_like(x[...,:1])],axis=-1)
    if op=='F':return x[...,::-1].copy()
    if op=='N':return ~x
    raise ValueError(op)

def execute(program,inputs):
    y=np.asarray(inputs,dtype=bool).copy()
    for op in program:y=step(y,op)
    return y

def pretty(program):return ' → '.join(program) if program else 'IDENTITY'
def truth_table(program):return execute(program,ALL_INPUTS).astype(np.uint8).tobytes()
PROGRAMS=[p for n in range(6) for p in product(TOKENS,repeat=n)]
assert len(PROGRAMS)==1365
assert np.array_equal(execute(('F','F'),ALL_INPUTS),ALL_INPUTS)
assert not np.array_equal(execute(('R','N'),ALL_INPUTS),
                          execute(('N','R'),ALL_INPUTS))
fig,axs=plt.subplots(1,3,figsize=(11,3.1),layout='constrained')
sample=ALL_INPUTS[[1,37,51,93]]
for ax,(p,title) in zip(axs,[((),'Input'),(('R','N'),'R then N'),(('N','R'),'N then R')]):
    ax.imshow(execute(p,sample),cmap='Greys',vmin=0,vmax=1,aspect='auto')
    ax.set(xticks=range(WIDTH),yticks=range(len(sample)),title=title,xlabel='Cell index')
plt.show()

## 2 · Syntax is not semantics: build equivalence classes

A brute-force grammar generates duplicate behavior. For example F(F(x)) = x. If we split *syntax strings* randomly between training and testing, two different programs can leak the same function across splits.

We group all 1,365 programs by their **complete 128-input truth table** and keep the **first shortest** representative as a canonical synthetic task. This entire-domain oracle is available only because the toy input universe is finite; it is **not** an inference tool available to a real ARC solver.

**Predict:** Do the 1,365 expressions imply 1,365 distinct transformations?

In [ ]:
groups={}
for p in PROGRAMS:
    key=truth_table(p)
    groups.setdefault(key,[]).append(p)
CANONICAL=[equivalents[0] for equivalents in groups.values()]
assert len(CANONICAL)<len(PROGRAMS)
assert len({truth_table(p) for p in CANONICAL})==len(CANONICAL)

fig,axs=plt.subplots(1,2,figsize=(11,3.5),layout='constrained')
for names,seqs in [('All syntax',PROGRAMS),('Distinct semantics',CANONICAL)]:
    count=np.bincount([len(p) for p in seqs],minlength=6)
    axs[0].plot(range(6),count,marker='o',label=names)
axs[0].set(xlabel='Minimal or literal program length',ylabel='Count',
           title='Many syntax trees implement the same function')
axs[0].legend()
size_hist=Counter(map(len,groups.values()))
axs[1].bar(sorted(size_hist),[size_hist[k] for k in sorted(size_hist)])
axs[1].set(xlabel='Syntactic programs per semantic family',
           ylabel='Semantic families',title='Equivalence-class sizes')
plt.show()
print('Raw programs:',len(PROGRAMS),'| exact semantic families:',len(CANONICAL))

## 3 · Four separate generalization questions

**IID-like held-out functions:** same short lengths and same allowed transition families, but different exact target transformations from those used to fit priors.

**Unseen composition:** a transition **R → N** is missing from the training solved-program traces, although **R** and **N** are individually known; evaluation targets contain it.

**Longer composition:** programs with **five** primitive steps when training sees at most **three**, with no held-out R → N transition.

**Joint shift:** five-step programs also containing R → N.

These are different generalization demands. A held-out action pair is a **distributional shift**, not a new primitive; a new primitive missing from the grammar is an **expressivity failure**, which we test separately.

We place a small **support set** of six solved, four-step R → N programs in a separate pool. It will be used for few-shot adaptation, never for final scoring. The support labels are **given**: acquiring solved support tasks has a real cost in practice.

**Predict:** If the policy has never observed R → N, can a smoothed policy still assign nonzero probability to that transition?

In [ ]:
def has_RN(p):return ('R','N') in tuple(zip(p,p[1:]))

split_rng=np.random.default_rng(SEED)
short=[p for p in CANONICAL if len(p)<=3 and not has_RN(p)]
split_rng.shuffle(short)
cut=int(.75*len(short))
TRAIN=short[:cut]
IID=short[cut:]

composition=[p for p in CANONICAL if len(p)==4 and has_RN(p)]
split_rng.shuffle(composition)
SUPPORT=composition[:6]
COMPOSITION=composition[6:]
LENGTH=[p for p in CANONICAL if len(p)==5 and not has_RN(p)]
JOINT=[p for p in CANONICAL if len(p)==5 and has_RN(p)]

SPLITS={'Familiar depth':IID,'New transition':COMPOSITION,
        'Longer programs':LENGTH,'Both shifts':JOINT}
all_sets=[TRAIN,SUPPORT,*SPLITS.values()]
all_programs=sum((list(s) for s in all_sets),[])
assert len(all_programs)==len(set(all_programs))
assert len({truth_table(p) for p in all_programs})==len(all_programs)
assert all(not has_RN(p) and len(p)<=3 for p in TRAIN)
assert all(has_RN(p) and len(p)==4 for p in SUPPORT+COMPOSITION)
assert all(len(p)==5 for p in LENGTH+JOINT)
assert set(''.join(''.join(p) for p in TRAIN))==set(TOKENS)

fig,ax=plt.subplots(figsize=(10,3.8))
sizes=[len(TRAIN),len(SUPPORT)]+[len(v) for v in SPLITS.values()]
names=['Train','Support']+list(SPLITS)
ax.bar(range(len(names)),sizes)
ax.set(xticks=range(len(names)),xticklabels=names,ylabel='Distinct semantic tasks',
       title='Disjoint groups defined before any policy adaptation')
ax.tick_params(axis='x',rotation=20)
plt.show()
print({k:len(v) for k,v in [('train',TRAIN),('support',SUPPORT),*SPLITS.items()]})

## 4 · Visualize the exact distribution shift

The training solver observes operations **and adjacent pairs** from solved programs. Count the transitions in the training set and show how R → N was deliberately withheld.

This is not a task-conditional policy yet. It models only the *language of likely solution programs*, analogous to a global prior over grammatical productions. The user-provided demonstrations will be checked later by the **exact interpreter**, not fed into this prior.

A neural version could condition its action distribution on the task input/output grids, partial program, source phenotype, depth, and search budget. The present baseline isolates what can—and cannot—be learned from grammar frequency alone.

**Predict:** Will an unobserved transition have zero probability under simple relative frequencies? Under Laplace smoothing?

In [ ]:
def transition_counts(tasks):
    counts=np.zeros((5,4),dtype=float)
    for p in tasks:
        prev=4
        for a in p:
            j=TO_IDX[a]
            counts[prev,j]+=1
            prev=j
    return counts

counts_train=transition_counts(TRAIN)
counts_support=transition_counts(SUPPORT)
fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
for ax,counts,title in zip(axs,[counts_train,counts_support],
                           ['Training solved programs','Six adaptation examples']):
    im=ax.imshow(counts,cmap='Blues',aspect='auto',interpolation='nearest')
    ax.set(xticks=range(4),xticklabels=TOKENS,
           yticks=range(5),yticklabels=list(TOKENS)+['START'],
           xlabel='Next operation',ylabel='Previous operation',title=title)
    for i in range(5):
        for j in range(4):ax.text(j,i,str(int(counts[i,j])),ha='center',va='center',
                                  color='white' if counts[i,j]>counts.max()/2 else 'black',fontsize=9)
fig.colorbar(im,ax=axs.ravel().tolist(),shrink=.75)
plt.show()
assert counts_train[TO_IDX['R'],TO_IDX['N']]==0
assert counts_support[TO_IDX['R'],TO_IDX['N']]>0

## 5 · Fit a smoothed policy prior

For each preceding operation \(a_{t-1}\), estimate the next-operation distribution

\[
\widehat P(a_t\mid a_{t-1})=
\frac{C(a_{t-1},a_t)+\alpha}
{\sum_b C(a_{t-1},b)+4\alpha}.
\]

Here \(\alpha>0\) supplies **smoothing**, giving all syntactically legal operations positive mass—even transitions not represented in the training corpus. This is a learned **Markov grammar prior**, not a posterior about which program fits a particular task.

We will use an intentionally *heuristic* candidate ranking:

\[
S(p)=\frac{1}{\max(1,|p|)}\sum_t\log \widehat P(p_t\mid p_{t-1})-0.15\,|p|.
\]

Average log probability removes most of the automatic penalty for longer programs, while the separate \(0.15|p|\) term expresses a weak length preference. **This score is not a normalized probability over complete programs.** The purpose is to rank candidates while keeping length extrapolation possible.

**Predict:** Could the learned prior favor incorrect programs? Can it assign zero mass to a legal but unseen composition?

In [ ]:
ALPHA=1.
def fit_prior(tasks,alpha=ALPHA):
    return transition_counts(tasks)+alpha
def get_probabilities(counts):
    return counts/counts.sum(axis=1,keepdims=True)
def program_score(p,counts,length_penalty=.15):
    if len(p)==0:return -3.
    probs=get_probabilities(counts)
    prev=4;score=0.
    for token in p:
        nxt=TO_IDX[token]
        score+=np.log(probs[prev,nxt])
        prev=nxt
    return score/len(p)-length_penalty*len(p)

BASE_COUNTS=fit_prior(TRAIN)
BASE_PROBS=get_probabilities(BASE_COUNTS)
assert np.all(BASE_PROBS>0)
assert np.allclose(BASE_PROBS.sum(axis=1),1)
fig,ax=plt.subplots(figsize=(8.5,3.4))
ax.bar(TOKENS,BASE_PROBS[TO_IDX['R']])
ax.set(xlabel='Action after R',ylabel='Smoothed conditional probability',
       title='An unseen legal transition stays possible')
plt.show()
print('P(N after R), before adaptation:',round(float(BASE_PROBS[TO_IDX['R'],TO_IDX['N']]),4))

## 6 · Rank complete candidate programs without peeking at their outputs

Construct the *full syntactic search space* of 1,365 programs, **including semantic duplicates**. We deliberately keep duplicate expressions because the cost of evaluating them is part of the search problem.

The ranking uses only the learned transition counts and cheap syntax features. No target-grid execution is used to create it. A uniform baseline shuffles the same candidate list.

This separates two budgets:

1. **Search-order preparation:** generating and sorting candidate ASTs and scoring syntactic priors.
2. **Candidate execution:** applying one candidate to task demonstrations and comparing outputs.

We later plot *candidate executions*, not wall time. A real deployment must also measure enumeration, model inference, bookkeeping, and cache cost.

**Predict:** If a five-step target's ranking is low under the training prior, how does that limit success at a fixed execution budget?

In [ ]:
def ranked_candidates(counts):
    scores=np.array([program_score(p,counts) for p in PROGRAMS])
    return np.argsort(-scores,kind='stable')

ORDER_BASE=ranked_candidates(BASE_COUNTS)
PROGRAM_INDEX={p:i for i,p in enumerate(PROGRAMS)}
def rank_of(p,order):
    indices=np.empty(len(order),dtype=int)
    indices[order]=np.arange(1,len(order)+1)
    return int(indices[PROGRAM_INDEX[p]])

fig,axs=plt.subplots(1,2,figsize=(11,3.5),layout='constrained')
for label,group in SPLITS.items():
    axs[0].scatter([len(p) for p in group],[rank_of(p,ORDER_BASE) for p in group],
                   s=19,alpha=.6,label=label)
axs[0].set(xlabel='Canonical target length',ylabel='Syntactic candidate rank',
           title='A training prior can bury shifted programs')
axs[0].legend(fontsize=7)
axs[1].hist([rank_of(p,ORDER_BASE) for p in JOINT],bins=14)
axs[1].set(xlabel='Candidate rank',ylabel='Joint-shift target count',
           title='Where are length-five R → N targets?')
plt.show()
assert len(ORDER_BASE)==len(PROGRAMS)
assert len(set(map(int,ORDER_BASE)))==len(PROGRAMS)

## 7 · Few-shot adaptation: update a prior using separate solved tasks

We are given a small support set of **already solved** tasks from the new distribution. Add their transition counts to the grammar prior with an adaptation weight \(\lambda\):

\[C_{\mathrm{adapted}}=C_{\mathrm{trained}}+\lambda\,C_{\mathrm{support}}.\]

This is **distribution adaptation from labeled support solutions**, not online gradient fine-tuning, not learning from the test answers, and not adaptation to the *current query's* demonstrations. The support solutions and evaluation programs belong to disjoint semantic families.

The weight is a teaching hyperparameter fixed here; in serious evaluation, tune it on validation tasks only. Large \(\lambda\) may overfit the support distribution and harm other tasks.

**Predict:** Should the transition R → N become more prominent? Could a better adapted prior still be worse on familiar tasks?

In [ ]:
ADAPT_WEIGHT=6.
ADAPTED_COUNTS=BASE_COUNTS+ADAPT_WEIGHT*transition_counts(SUPPORT)
ADAPTED_PROBS=get_probabilities(ADAPTED_COUNTS)
ORDER_ADAPT=ranked_candidates(ADAPTED_COUNTS)
r=TO_IDX['R'];n=TO_IDX['N']
print('P(N | R):',round(float(BASE_PROBS[r,n]),4),'→',
      round(float(ADAPTED_PROBS[r,n]),4))
fig,axs=plt.subplots(1,2,figsize=(11,3.5),layout='constrained')
xx=np.arange(4)
axs[0].bar(xx-.2,BASE_PROBS[r],width=.4,label='Before')
axs[0].bar(xx+.2,ADAPTED_PROBS[r],width=.4,label='After 6 support solves')
axs[0].set(xticks=xx,xticklabels=TOKENS,ylim=(0,1),ylabel='P(next | R)',
           title='Adaptation changes the action prior')
axs[0].legend(fontsize=8)
base_rank=np.array([rank_of(p,ORDER_BASE) for p in JOINT])
adapt_rank=np.array([rank_of(p,ORDER_ADAPT) for p in JOINT])
axs[1].scatter(base_rank,adapt_rank)
bound=max(base_rank.max(),adapt_rank.max())
axs[1].plot([0,bound],[0,bound],ls='--',color='gray')
axs[1].set(xlabel='Rank before adaptation',ylabel='Rank after adaptation',
           title='R → N joint-shift program ranking')
plt.show()
assert ADAPTED_PROBS[r,n]>BASE_PROBS[r,n]

## 8 · What does it mean to *solve* a synthesis task?

The search agent receives **a few demonstration pairs** generated by an unknown target program \(p^*\):

\[\mathcal D=\{(x_i,p^*(x_i))\}_{i=1}^m.\]

It executes candidate programs in ranked order. It **stops at the first program that matches every demonstration**. Afterwards, an independent evaluator compares that proposed program with the true function over the entire 128-input toy domain.

We therefore track three distinct outcomes:

- **No candidate found:** the budget runs out.
- **Demonstration fit but incorrect function:** an *overfit/spurious* solution.
- **Exact semantic success:** the proposed program matches the target function everywhere in this finite toy world.

No hidden oracle output is consulted **while choosing candidates or deciding to stop**. The evaluator uses it *only after* the candidate has been selected.

**Predict:** Can adding demonstration examples reduce spurious solves without changing the grammar or the candidate order?

In [ ]:
def propose(target,order,demo_ids,budget=350):
    inputs=ALL_INPUTS[np.asarray(demo_ids,dtype=int)]
    outputs=execute(target,inputs)
    tries=0
    for index in order[:budget]:
        candidate=PROGRAMS[int(index)]
        tries+=1
        if np.array_equal(execute(candidate,inputs),outputs):
            # The oracle here is EVALUATION ONLY, after the solver has stopped.
            true_function=np.array_equal(execute(candidate,ALL_INPUTS),
                                         execute(target,ALL_INPUTS))
            return dict(found=True,exact=bool(true_function),calls=tries,
                        candidate=candidate)
    return dict(found=False,exact=False,calls=tries,candidate=None)

demo_rng=np.random.default_rng(101)
example=JOINT[0]
ids=demo_rng.choice(len(ALL_INPUTS),size=3,replace=False)
b=propose(example,ORDER_BASE,ids)
a=propose(example,ORDER_ADAPT,ids)
print('Hidden target (only for our diagnostics):',pretty(example))
print('Training prior:',b)
print('Adapted prior:',a)
assert b['calls']<=350 and a['calls']<=350
assert not a['exact'] or a['found']

## 9 · Make the three kinds of outcome visible

Inspect a single query with a few observed demonstrations. If a solver returns a program matching those examples, it has **established consistency**, not logically established the intended transformation.

The visual shows the selected demonstrations, the hidden target outputs, and the program chosen by the adapted search. The hidden-target row belongs to **diagnostics** and is never used in candidate ranking.

**Predict:** If two different programs produce identical outputs on the demos, how could their performance diverge on unseen inputs?

In [ ]:
target=JOINT[3]
ids=np.random.default_rng(220).choice(128,3,replace=False)
record=propose(target,ORDER_ADAPT,ids,budget=350)
inputs=ALL_INPUTS[ids]
chosen=record['candidate']
fig,axs=plt.subplots(1,3,figsize=(11,2.7),layout='constrained')
to_display=[inputs,execute(target,inputs),
            execute(chosen,inputs) if chosen is not None else np.zeros_like(inputs)]
for ax,arr,title in zip(axs,to_display,['Demonstration inputs','Target outputs','Selected candidate outputs']):
    ax.imshow(arr,cmap='Greys',vmin=0,vmax=1,aspect='auto')
    ax.set(xticks=range(WIDTH),yticks=range(3),xlabel='Grid position',title=title)
plt.show()
print('Selected candidate:',pretty(chosen) if chosen is not None else '<none>')
print('Consistent:',record['found'],'| exactly equivalent on entire domain:',record['exact'])

## 10 · Benchmark fair allocation across disjoint groups

Compare:
- **Uniform shuffled enumeration** (same grammar, randomized order).
- **Pretrained Markov prior** (from short training solutions).
- **Few-shot-adapted prior** (same prior plus the six *separate* solved support tasks).

All methods see the **same three demonstration inputs for each query**, the same complete program grammar, and at most **350 exact candidate executions**. No method is allowed to use the hidden full-domain truth table for search decisions.

For uniform enumeration we average several random order seeds to reduce accidental luck. The other policies use deterministic ordering with stable ties. Evaluation uses fixed task identities and demo seeds—never re-trains on the final groups.

**Predict:** Which method should be strongest for the intentionally shifted R → N families? Does that logically imply it will also win on length-only tasks?

In [ ]:
BUDGET=350
METHODS=['Uniform enumeration','Trained prior','Adapted prior']
def evaluate_group(targets,order,budget=BUDGET,n_demos=3,seed=111):
    results=[]
    for i,target in enumerate(targets):
        ids=np.random.default_rng(seed+31*i).choice(128,n_demos,replace=False)
        results.append(propose(target,order,ids,budget=budget))
    return results
def summary(rows):
    if not rows:return (0.,0.,np.nan)
    return (float(np.mean([r['found'] for r in rows])),
            float(np.mean([r['exact'] for r in rows])),
            float(np.median([r['calls'] for r in rows])))

BENCH={}
for group_name,targets in SPLITS.items():
    shuffle_results=[]
    for repetition in range(3):
        order=np.random.default_rng(1000+repetition).permutation(len(PROGRAMS))
        shuffle_results.extend(evaluate_group(targets,order,seed=111))
    BENCH[group_name]={
        'Uniform enumeration':shuffle_results,
        'Trained prior':evaluate_group(targets,ORDER_BASE,seed=111),
        'Adapted prior':evaluate_group(targets,ORDER_ADAPT,seed=111)
    }
for group_name in SPLITS:
    print(group_name)
    for method in METHODS:
        found,exact,median=summary(BENCH[group_name][method])
        print(f'  {method:20s} demo fit={found:.3f} exact={exact:.3f} median calls={median:.0f}')

fig,ax=plt.subplots(figsize=(11,4.5))
x=np.arange(len(SPLITS))
for j,method in enumerate(METHODS):
    score=[summary(BENCH[g][method])[1] for g in SPLITS]
    ax.bar(x+(j-1)*.24,score,width=.23,label=method)
ax.set(xticks=x,xticklabels=list(SPLITS),ylim=(0,1.05),
       ylabel='Exact semantic success fraction',title='Equal candidate execution budgets')
ax.legend(fontsize=8,ncol=3);plt.show()

## 11 · Discovery curves: a fixed budget is a scientific constraint

A policy can look good at 350 executions yet poor at 50, or vice versa. We draw exact solve fraction **as a function of allowed candidate executions**, based on each query's first consistent result.

A wrong early solution is especially important: once the solver stops at that consistent-but-spurious program, giving it a larger execution budget **does not repair** that run. More demonstrations or an explicit holdout check are needed to distinguish alternatives.

The curves use the same runs as the benchmark, and therefore do not consume new query information.

**Predict:** Why can an apparently successful demonstration-fitting method remain below 100% even after many extra evaluations?

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for ax,grp in zip(axs,['New transition','Both shifts']):
    for method in METHODS:
        rows=BENCH[grp][method]
        ys=[np.mean([r['exact'] and r['calls']<=b for r in rows])
            for b in range(1,BUDGET+1)]
        ax.plot(np.arange(1,BUDGET+1),ys,label=method)
    ax.set(xlabel='Allowed candidate executions',ylabel='Exact success fraction',
           title=grp,ylim=(0,1.03))
    ax.legend(fontsize=8)
plt.show()

## 12 · Negative transfer is visible, not a footnote

Adaptation is not automatically progress on *every* task. Updating a global grammar prior toward one newly observed family can reorder the search unfavorably for another family.

Compare the **adapted-minus-pretrained** exact solve rates for the same demonstrations. A positive bar indicates improvement on that synthetic group; a negative bar indicates negative transfer.

This is why support and query distributions matter. A single global policy may be the wrong abstraction when different industries, task types, source phenotypes, or object decompositions demand different operation distributions.

**Predict:** Would task-conditioned mixture-of-experts priors help if contexts reliably identify the current family? What could go wrong under an unfamiliar family?

In [ ]:
names=list(SPLITS)
delta=np.array([summary(BENCH[g]['Adapted prior'])[1]-
                summary(BENCH[g]['Trained prior'])[1] for g in names])
fig,ax=plt.subplots(figsize=(10,3.8))
ax.bar(names,delta,color=['#78b695' if x>=0 else '#cc8d8d' for x in delta])
ax.axhline(0,color='gray')
ax.set(ylabel='Adapted minus trained exact solve rate',
       title='Transfer and negative transfer coexist')
ax.tick_params(axis='x',rotation=17)
plt.show()
print(dict(zip(names,np.round(delta,3))))

## 13 · How many solved support tasks are worth learning from?

Use the same support pool but expose only 0, 1, 3, or 6 solved examples. The test tasks remain fixed and semantically distinct from all support tasks.

The update is deliberately cheap—increment a transition-count table—but the **support solutions are not free**. In a real application, obtaining verified programs may require human effort, solver runtime, or scarce labeled data. A cost-benefit analysis must include those expenses.

We hold the adaptation weight fixed across support sizes to isolate the effect of additional examples. Small support sets can change the prior dramatically and therefore have high variance.

**Predict:** Should support count always improve the joint-shift exact solve rate monotonically?

In [ ]:
SUPPORT_SIZES=[0,1,3,6]
few_shot={}
for k in SUPPORT_SIZES:
    counts=BASE_COUNTS+ADAPT_WEIGHT*transition_counts(SUPPORT[:k])
    order=ranked_candidates(counts)
    few_shot[k]={g:summary(evaluate_group(SPLITS[g],order,seed=111))[1]
                 for g in ('New transition','Both shifts','Longer programs')}
fig,ax=plt.subplots(figsize=(9.2,3.7))
for group in ('New transition','Both shifts','Longer programs'):
    ax.plot(SUPPORT_SIZES,[few_shot[k][group] for k in SUPPORT_SIZES],
            marker='o',label=group)
ax.set(xticks=SUPPORT_SIZES,xlabel='Number of solved support tasks',
       ylabel='Exact semantic success fraction',ylim=(0,1.03),
       title='Few-shot adaptation can help one family and hurt another')
ax.legend();plt.show()

## 14 · Fitting examples is not proving generalization

Reduce or increase the number of demonstrations while leaving the candidate ordering and budget unchanged. With too few observations, an incorrect program may happen to agree on all observed inputs.

Our **external truth-table oracle** measures whether the found candidate is actually the same function, but in a real task the unknown true behavior on unseen inputs cannot be checked exhaustively.

This motivates active test selection (lesson 03's information gain), uncertainty-aware verification, robust split design, and collecting counterexamples that specifically distinguish competing hypotheses.

**Predict:** If a candidate already matches a single example, can its posterior correctness be inferred from that fact alone?

In [ ]:
demo_sizes=[1,2,3,6,12]
fig,ax=plt.subplots(figsize=(9.1,3.7))
seen_success=[];verified_success=[]
for n in demo_sizes:
    runs=evaluate_group(JOINT,ORDER_ADAPT,n_demos=n,seed=555)
    seen_success.append(summary(runs)[0])
    verified_success.append(summary(runs)[1])
ax.plot(demo_sizes,seen_success,marker='o',label='Demonstration fit')
ax.plot(demo_sizes,verified_success,marker='o',label='Exact full-domain match')
ax.set(xticks=demo_sizes,xlabel='Observed task examples',
       ylim=(0,1.05),ylabel='Fraction of joint-shift tasks',
       title='Distinguish fitting a few examples from functional identity')
ax.legend();plt.show()
print('Demo counts:',demo_sizes)
print('Demonstration fits:',np.round(seen_success,3))
print('Full-domain exact matches:',np.round(verified_success,3))

## 15 · An unseen primitive is categorically different from a new combination

Our test up to now assumes the grammar contains all four primitives. The learned prior may dislike R → N, but both operations remain **legal**. With enough search, their composition can be attempted.

Now remove NOT from the grammar entirely and ask for the function NOT(INPUT). No reordering, UCB bonus, macro-prior update, or neural embedding can create a missing semantic operation **unless other allowed primitives can express it**. In this grammar, shifts and reversal cannot perform complementation.

We demonstrate the difference with exhaustive execution on the finite toy domain; the point is one about **hypothesis-class expressivity**, not the quality of a search policy.

**Predict:** Can unlimited compute rescue a solver whose language lacks any way to change a false input bit into true under these permitted operations?

In [ ]:
WITHOUT_NOT=[p for p in PROGRAMS if 'N' not in p]
target=('N',)
has_equivalent=any(np.array_equal(execute(p,ALL_INPUTS),execute(target,ALL_INPUTS))
                   for p in WITHOUT_NOT)
assert not has_equivalent
fig,ax=plt.subplots(figsize=(7.8,3.4))
ax.bar(['Original grammar','Without NOT'],[len(PROGRAMS),len(WITHOUT_NOT)])
ax.set(ylabel='Number of syntactically legal programs',
       title='More search cannot recover a missing required operator')
plt.show()
print('NOT(INPUT) expressible with R, L, F only?',has_equivalent)

## 16 · Variation, uncertainty, and the danger of cherry-picking

A single accuracy fraction for a group of 8 or 37 intentionally selected programs is not a general-population performance estimate. **Wilson intervals** summarize binomial uncertainty under hypothetical independent task sampling, but our groups are *constructed*, not independent random draws from all ARC-like problems. Interpret intervals as descriptive, not a certified ARC confidence statement.

For randomized enumeration, selection depends on the random program order; for trained/adapted priors, task selection and demonstrations are the key sources of variation. A stronger study would bootstrap **task identities**, repeat seeds, freeze all model choices before final evaluation, and publish both successes and failures.

**Predict:** Which is less informative: 7/8 wins on a curated toy family, or 70/80 wins on independently sampled, externally defined held-out tasks?

In [ ]:
def wilson_interval(success,n,z=1.96):
    if n==0:return (np.nan,np.nan)
    phat=success/n;zz=z*z
    center=(phat+zz/(2*n))/(1+zz/n)
    radius=z*np.sqrt(phat*(1-phat)/n+zz/(4*n*n))/(1+zz/n)
    return max(0,center-radius),min(1,center+radius)

fig,ax=plt.subplots(figsize=(9,3.7))
groups_to_show=['New transition','Longer programs','Both shifts']
for j,g in enumerate(groups_to_show):
    vals=[]
    for method in ('Trained prior','Adapted prior'):
        records=BENCH[g][method]
        n=len(records);success=sum(r['exact'] for r in records)
        lo,hi=wilson_interval(success,n)
        vals.append((success/n,lo,hi))
    for k,(p,lo,hi) in enumerate(vals):
        x=j+(-.12 if k==0 else .12)
        ax.errorbar(x,p,yerr=[[p-lo],[hi-p]],fmt='o',capsize=4,
                    label=('Trained' if k==0 else 'Adapted') if j==0 else None)
ax.set(xticks=range(len(groups_to_show)),xticklabels=groups_to_show,
       ylim=(0,1.03),ylabel='Exact fraction / descriptive Wilson interval',
       title='Sample sizes and uncertainty matter')
ax.legend();plt.show()

## 17 · Engineering the experiment for a serious Grammar-UCT / neural solver

In your symbolic system, the action is not just a letter: it can be a **typed operation + source phenotype + concrete source + parameters**, and a physical gene may bundle many semantic components. A suitable evaluation harness should keep these levels distinct.

A defensible workflow is:

1. **Freeze task partitions** by semantic families, dataset origin, and difficulty; do not let variants of the same transformation appear across train/test.
2. **Fit reusable priors, representations, or macros** using only training solves, including unsuccessful-task logs if available.
3. **Choose hyperparameters** on a separate validation set.
4. **Adapt** only from explicitly permitted, already solved support tasks, and report their acquisition cost.
5. **Search query tasks** using the given demonstrations; count attempted actions, valid candidates, duplicate discards, accepted unique genes, exact candidate executions, and wall time **separately**.
6. **Verify proposed programs** on observed demonstrations; assess generalization only using the held-out evaluator.
7. **Report exact solutions, budgeted discovery curves, failure classes, intervals, and negative transfer** across multiple fixed seeds.

A helpful task context embedding may separate operator distributions and avoid the nonconditional bigram baseline's negative transfer, but learned embeddings can also mislead under new compositions. Preserve a uniform-exploration fallback.

**Central lesson:** *a stronger model representation is useful only insofar as it improves decisions on genuinely unseen tasks at a controlled cost.*

## 18 · What this says about self-supervised target discovery and research

This series has repeatedly separated **finding a model that fits** from **understanding what kinds of solutions search prefers**:

- A grammar selects which transformations are expressible.
- A prior changes which expressive programs are visited first.
- A reward or likelihood changes which observed programs appear preferable.
- An embedding determines which contexts or behaviors can transfer between tasks.
- A library can make a useful composition cheap, but it can also enlarge the branching factor.
- Adaptation may rapidly improve a new domain while reducing performance in another.

In self-supervised scientific discovery, these interactions matter because the target itself can be selected from many plausible alternatives. A model that confidently fits familiar targets may still generalize poorly when the objective or data-generating process changes.

For a real research claim, distinguish empirical discovery under your **defined evaluation protocol** from claims of general causal structure, universal grammar optimality, or out-of-distribution robustness. Report uncertainty over targets **and** over the search/selection process.

**What to study next:** meta-learning and fast adaptation, amortized program synthesis, minimum-description-length grammars, counterexample-guided inductive synthesis (CEGIS), distributionally robust optimization, and structure-aware policy/value learning.

## 19 · Exercises — reconstruct and improve the benchmark

1. **Equivalence leakage:** Explain why splitting the 1,365 raw syntax strings at random would invalidate a claim about transfer to unseen semantic functions.
2. **True composition split:** Replace R → N with a different withheld legal pair. Identify whether both components are represented in training.
3. **Smoothing:** Set \(\alpha\) to 0.1, 1, and 10. Plot how the unobserved pair probability and test success respond.
4. **Ranking objective:** Replace the average token log score with true autoregressive log probability plus an explicit termination model. Compare length-five discovery.
5. **Few-shot support:** Vary the adaptation weight \(\lambda\) without touching the test labels. Which validation split should select it?
6. **Spurious programs:** Find a task for which the first candidate matches two examples but disagrees on the complete domain. Choose a third input that distinguishes them.
7. **Active query:** Implement a small information-gain heuristic that chooses the next demonstration input where remaining candidate programs disagree most.
8. **Grammar limitation:** Remove operation F. Identify any canonical target families that are no longer representable.
9. **Semantics-aware deduplication:** Canonicalize programs before proposing them, but explicitly account for the executions needed to learn their equivalence classes. Can it be fairly called "free"?
10. **Context-conditioned adaptation:** Train a small policy that predicts operations from input/output demonstrations. Compare a fixed prior, conditioned model, and few-shot-adapted prior on the same disjoint task groups.
11. **Library transfer:** Admit a macro learned in 07 only from training tasks and quantify benefit/negative transfer on the same evaluation groups.
12. **Statistical protocol:** Pre-register a primary metric, a compute budget, supported adaptation data, and a rule for selecting hyperparameters without using the final test group.

Take special care not to use held-out truth tables to build the candidate ranking.

## 20 · Worked solutions and synthesis

1. Equivalent programs leak behavior despite different surface syntax. Split semantic classes—or a properly grounded real-world task-family analogue—before model or library training.
2. Generate canonical tasks, filter the transition in training, verify both primitives remain present, and maintain disjoint behavior fingerprints.
3. Small \(\alpha\) makes observed counts dominate; large \(\alpha\) approaches uniform transitions. Either may improve or degrade fixed-budget search.
4. Proper program probability needs termination and length modeling; raw sums systematically favor short programs, while averages are not normalized distributions.
5. Use validation tasks from the anticipated support/query shift and lock \(\lambda\) before examining final test outcomes.
6. Enumerate the candidate/target output disagreement over the 128-input domain **only in this toy test**. A discriminating input establishes a counterexample.
7. Score candidate input probes by diversity or entropy of predicted outputs across the remaining consistent set. Selecting probes changes the experimental observation process and its cost.
8. Test expressivity by exact semantic equivalence over the finite toy domain; no search policy can rescue a transformation absent from the reduced grammar.
9. A real equivalence check requires executions/proofs; hiding that work makes budget comparisons invalid.
10. Hold out entire semantic families, not task rows. Report exact solve fractions and candidate execution counts; a learned conditional policy may generalize or overfit.
11. Library entries have definition/branch overhead. Charge macro construction and training-solves once, then separately report amortized query costs.
12. Use held-out task identities, fixed random seeds, equal execution/runtime budgets, and a final evaluation run not repeatedly tuned against its own results.

### Reconstruct without looking

You should be able to implement an exact tiny grammar, semantic-equivalence grouping, disjoint length/composition task splits, smoothed action prior, few-shot support update, exact candidate search, demo-fit versus true-function evaluation, matched-budget curves, and a missing-primitive sanity check.

### Primary reading and connections

- Lake & Baroni (2018), *Generalization without Systematicity: On the Compositional Skills of Sequence-to-Sequence Recurrent Networks* (SCAN).
- Kim & Linzen (2020), *COGS: A Compositional Generalization Challenge Based on Semantic Interpretation*.
- Finn, Abbeel & Levine (2017), *Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks* (MAML).
- Ellis et al. (2021), *DreamCoder: Bootstrapping Inductive Program Synthesis with Wake-Sleep Library Learning*.
- Gulwani, Polozov & Singh (2017), *Program Synthesis* (survey).
- Solar-Lezama (2008), *Program Synthesis by Sketching*.

**You have completed the eight-plus-one lessons of vector_01 (00–08).** The next meaningful project is an experiment suite: implement one reproducible grammar-search benchmark with train/validation/test task families, compare uniform/UCT/neural priors, measure compute, then add embeddings and libraries one at a time. This lesson's source is deliberately a readable starting point, not benchmark evidence for ARC-AGI.